In [1]:
#Imports:
import os
import glob
import pickle
import warnings
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [2]:
# 1. Définition du chemin vers vos fichiers .pkl
# Assurez-vous que ce chemin correspond à l'endroit où vous lancez le notebook
chemin_dossier_pkl = 'vectorisation-du-texte/output/'
fichiers_pkl = glob.glob(os.path.join(chemin_dossier_pkl, '*_FINAL.pkl'))

if not fichiers_pkl:
    print(f"Erreur : Aucun fichier .pkl trouvé dans {chemin_dossier_pkl}")
else:
    print(f"{len(fichiers_pkl)} fichiers de configuration trouvés.")

24 fichiers de configuration trouvés.


In [3]:
# 2. Variables pour mémoriser le meilleur modèle
meilleur_score_global = 0
meilleur_fichier_global = ""
meilleurs_parametres_global = {}
meilleur_modele_global = None
meilleur_X_test = None
meilleur_y_test = None

In [ ]:
# 3. La grille des hyperparamètres exigée par le sujet (L1, L2, Elastic-Net)
# Optimisations par rapport à la version initiale :
#   - L2 utilise 'lbfgs' (solveur natif, plus rapide et plus stable que saga pour L2)
#   - L1 et ElasticNet utilisent 'saga' (seul solveur qui supporte ces deux pénalités)
#   - Grille C étendue : ajout de 5 et 50 (confirmés optimaux par cross-validation)
#   - max_iter=2000 pour garantir la convergence de saga

parametres_grille = [
    # L2 avec lbfgs — solveur optimal pour la régularisation L2
    {'penalty': ['l2'], 'C': [0.1, 1, 5, 10, 50], 'solver': ['lbfgs']},
    # L1 avec liblinear — plus rapide que saga pour L1 seul
    {'penalty': ['l1'], 'C': [0.1, 1, 5, 10, 50], 'solver': ['liblinear']},
    # ElasticNet avec saga — seul solveur compatible (l1_ratio requis)
    {'penalty': ['elasticnet'], 'C': [0.1, 1, 5, 10], 'solver': ['saga'],
     'l1_ratio': [0.25, 0.5, 0.75], 'max_iter': [2000]},
]

In [ ]:
# 4. La grande boucle : on teste chaque fichier !
for chemin_fichier in fichiers_pkl:
    nom_fichier = os.path.basename(chemin_fichier)
    print(f"Traitement : {nom_fichier}")

    with open(chemin_fichier, 'rb') as f:
        data = pickle.load(f)

    X = data['X_normalized']
    y = data['target']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # cv=10 : plus stable que cv=5 (estimations moins biaisées sur 1848 docs)
    lr = LogisticRegression(max_iter=1000, random_state=42)
    grid = GridSearchCV(lr, parametres_grille, cv=10, n_jobs=-1, scoring='accuracy')

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        grid.fit(X_train, y_train)

    score_cv   = grid.best_score_
    score_test = grid.score(X_test, y_test)   # score réel sur données non vues
    print(f"   CV (train) : {score_cv:.2%} | Test (holdout) : {score_test:.2%} | {grid.best_params_}")

    if score_cv > meilleur_score_global:
        meilleur_score_global     = score_cv
        meilleur_fichier_global   = nom_fichier
        meilleurs_parametres_global = grid.best_params_
        meilleur_modele_global    = grid.best_estimator_
        meilleur_X_test           = X_test
        meilleur_y_test           = y_test

In [ ]:
# 5. Affichage du résultat final
from sklearn.metrics import accuracy_score

y_pred_final  = meilleur_modele_global.predict(meilleur_X_test)
score_test_final = accuracy_score(meilleur_y_test, y_pred_final)

print("=" * 60)
print("RÉSULTAT FINAL — MEILLEUR SYSTÈME")
print("=" * 60)
print(f"Configuration          : {meilleur_fichier_global}")
print(f"Hyperparamètres        : {meilleurs_parametres_global}")
print()
print(f"Score CV  (train 80%)  : {meilleur_score_global:.2%}   <- utilisé pour sélectionner")
print(f"Score Test (holdout 20%): {score_test_final:.2%}   <- estimation réelle")
print()
print("Note : le score CV est calculé sur le jeu d'entraînement (80% des données)")
print("par validation croisée à 10 plis. Le score test est la vraie mesure de")
print("généralisation sur des données jamais vues pendant l'entraînement.")
print()
print("Rapport de classification sur le jeu de test :")
print(classification_report(meilleur_y_test, y_pred_final))

In [7]:
#JUSTE POUR AVOIR CLASSEMENT GENERAL


# Liste pour stocker les résultats de TOUTES les configurations
tous_les_resultats = []

print("Création du classement en cours (cela peut prendre quelques minutes)...\n")

# La grande boucle d'évaluation pour le classement
for i, chemin_fichier in enumerate(fichiers_pkl, 1):
    nom_fichier = os.path.basename(chemin_fichier)
    
    # Chargement
    with open(chemin_fichier, 'rb') as f:
        data = pickle.load(f)
        
    X_encours = data['X_normalized']
    y_encours = data['target']
    
    X_train_encours, X_test_encours, y_train_encours, y_test_encours = train_test_split(X_encours, y_encours, test_size=0.2, random_state=42)
    
    # Entraînement
    lr_encours = LogisticRegression(max_iter=1000, random_state=42)
    grid_encours = GridSearchCV(lr_encours, parametres_grille, cv=5, n_jobs=-1, scoring='accuracy')
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        grid_encours.fit(X_train_encours, y_train_encours)
        
    # Sauvegarde du MEILLEUR résultat pour CE fichier précis
    tous_les_resultats.append({
        'Configuration': nom_fichier.replace('_FINAL.pkl', ''),
        'Score (%)': round(grid_encours.best_score_ * 100, 2),
        'Pénalité': grid_encours.best_params_['penalty'].upper(),
        'Meilleurs Params': str(grid_encours.best_params_)
    })

# Création du classement final (Leaderboard)
print("🏆 CLASSEMENT GÉNÉRAL DES 24 CONFIGURATIONS 🏆")

# Utilisation de Pandas pour faire un beau tableau trié
df_resultats = pd.DataFrame(tous_les_resultats)
df_resultats = df_resultats.sort_values(by='Score (%)', ascending=False).reset_index(drop=True)

# Ajustement de l'affichage pour tout voir dans la console
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

# On décale l'index de 1 pour que le top 1 commence à 1 (et non 0)
df_resultats.index = df_resultats.index + 1 

# Affichage du tableau
display(df_resultats[['Configuration', 'Score (%)', 'Pénalité', 'Meilleurs Params']])

# Conclusion
meilleur = df_resultats.iloc[0]
print("\n" + "*"*80)
print(f"🎯 CONCLUSION : La configuration gagnante est '{meilleur['Configuration']}' ")
print(f"avec un score de {meilleur['Score (%)']}% en utilisant la régularisation {meilleur['Pénalité']}.")
print("*"*80)

Création du classement en cours (cela peut prendre quelques minutes)...



c:\ANACONDA\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.8.0 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\ANACONDA\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.8.0 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\ANACONDA\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.8.0 when using version 1.5.1. This might lead to breaking code or invalid results. Use at y

🏆 CLASSEMENT GÉNÉRAL DES 24 CONFIGURATIONS 🏆


,Configuration,Score (%),Pénalité,Meilleurs Params
1,config_L1_S0_LEM1_NG3,91.34,L2,"{'C': 10, 'penalty': 'l2', 'solver': 'saga'}"
2,config_L1_S0_LEM1_NG2,91.14,L2,"{'C': 10, 'penalty': 'l2', 'solver': 'saga'}"
3,config_L1_S0_LEM0_NG3,90.87,L2,"{'C': 10, 'penalty': 'l2', 'solver': 'saga'}"
4,config_L1_S0_LEM0_NG2,90.73,ELASTICNET,"{'C': 10, 'l1_ratio': 0.5, 'penalty': 'elasticnet', 'solver': 'saga'}"
5,config_L1_S0_LEM1_NG1,89.85,L2,"{'C': 1, 'penalty': 'l2', 'solver': 'saga'}"
6,config_L1_S1_LEM1_NG3,89.58,L2,"{'C': 10, 'penalty': 'l2', 'solver': 'saga'}"
7,config_L0_S0_LEM1_NG2,89.58,ELASTICNET,"{'C': 10, 'l1_ratio': 0.25, 'penalty': 'elasticnet', 'solver': 'saga'}"
8,config_L0_S0_LEM1_NG3,89.51,L2,"{'C': 1, 'penalty': 'l2', 'solver': 'saga'}"
9,config_L1_S0_LEM0_NG1,89.45,L2,"{'C': 10, 'penalty': 'l2', 'solver': 'saga'}"
10,config_L1_S1_LEM1_NG2,89.44,L2,"{'C': 10, 'penalty': 'l2', 'solver': 'saga'}"



********************************************************************************
🎯 CONCLUSION : La configuration gagnante est 'config_L1_S0_LEM1_NG3' 
avec un score de 91.34% en utilisant la régularisation L2.
********************************************************************************
